# 02 - Data Preprocessing and Feature Engineering (Member 2)
Input: the 8 Home Credit tables explored in `01_EDA.ipynb`.
Output: processed train/validation/test sets + a reusable preprocessing pipeline.

Plan
1. Load data
2. Apply the cleaning decisions from the EDA
3. Engineer application-level features
4. Aggregate the history tables to one row per client and join
5. Split train / validation (stratified) BEFORE learning anything
6. Fit the preprocessing pipeline on the training part only
7. Leakage checks, save outputs

## 1. Setup and data loading (local Jupyter)
Place this notebook in the same folder as `data/` and `src/`.

In [6]:
import os, sys
import numpy as np, pandas as pd

project = os.getcwd() + os.sep         # folder that contains this notebook, data/ and src/
sys.path.append(project)
path = project + "data" + os.sep

from src.features import build_features
from src.preprocessing import (split_xy, train_valid_split, build_preprocessor,
                               get_feature_names, transform_to_frame, save_pipeline)

In [7]:
train = pd.read_csv(path + "application_train.csv")
test = pd.read_csv(path + "application_test.csv")
bureau = pd.read_csv(path + "bureau.csv")
bureau_bal = pd.read_csv(path + "bureau_balance.csv")
prev = pd.read_csv(path + "previous_application.csv")
pos = pd.read_csv(path + "POS_CASH_balance.csv")
inst = pd.read_csv(path + "installments_payments.csv")
cc = pd.read_csv(path + "credit_card_balance.csv")
print(train.shape, test.shape)

(307511, 122) (48744, 121)


## 2. Cleaning decisions taken from the EDA
| EDA finding | Decision |
|---|---|
| `DAYS_EMPLOYED = 365243` (about 18% of rows) | replace with NaN and keep a flag `DAYS_EMPLOYED_ANOM` |
| `XNA` in `CODE_GENDER`, `ORGANIZATION_TYPE` | treat as missing |
| Housing features / `OWN_CAR_AGE` very sparse | drop columns with more than 60% missing (threshold learned on train only) |
| Outliers in income / credit / annuity | rows kept; they look like real values (use `scale=False` tree models or log transform if needed) |
| 8% defaulters | stratified split; handle imbalance in the modelling step |
| History tables have many rows per client | aggregate to one row per `SK_ID_CURR`; missing history gets a `HAS_..._HISTORY` flag |
| `365243` also exists in `previous_application` DAYS_* columns | replaced with NaN before aggregating |

## 3 and 4. Feature engineering (application features + history aggregates)

In [8]:
hist = dict(bureau=bureau, bureau_bal=bureau_bal, prev=prev, pos=pos, inst=inst, cc=cc)
train_fe = build_features(train, **hist)
test_fe = build_features(test, **hist)       # same code, same logic -> same columns
print(train_fe.shape, test_fe.shape)

(307511, 182) (48744, 181)


In [9]:
# sanity checks
assert len(train_fe) == len(train) and len(test_fe) == len(test)
assert train_fe["SK_ID_CURR"].is_unique and test_fe["SK_ID_CURR"].is_unique
assert set(train_fe.columns) - set(test_fe.columns) == {"TARGET"}
new_cols = [c for c in train_fe.columns if c not in train.columns]
print(len(new_cols), "new columns")
train_fe[new_cols].describe().T.head(40)

60 new columns


,count,mean,std,min,25%,50%,75%,max
DAYS_EMPLOYED_ANOM,307511.0,1.800716e-01,3.842477e-01,0.000000e+00,0.000000,0.000000,0.000000e+00,1.000000e+00
AGE_YEARS,307511.0,4.393697e+01,1.195613e+01,2.051781e+01,34.008219,43.150685,5.392329e+01,6.912055e+01
YEARS_EMPLOYED,252137.0,6.531971e+00,6.406466e+00,-0.000000e+00,2.101370,4.515068,8.698630e+00,4.907397e+01
EMPLOYED_TO_AGE_RATIO,252137.0,1.568615e-01,1.335486e-01,-0.000000e+00,0.056099,0.118733,2.191704e-01,7.288115e-01
CREDIT_INCOME_RATIO,307511.0,3.957570e+00,2.689728e+00,4.807615e-03,2.018667,3.265067,5.159880e+00,8.473684e+01
ANNUITY_INCOME_RATIO,307499.0,1.809300e-01,9.457380e-02,2.238846e-04,0.114782,0.162833,2.290667e-01,1.875965e+00
CREDIT_TERM,307499.0,2.161232e+01,7.823823e+00,8.036674e+00,15.614496,20.000000,2.709998e+01,4.530508e+01
GOODS_CREDIT_RATIO,307233.0,9.006893e-01,9.663036e-02,1.666667e-01,0.834725,0.893815,1.000000e+00,6.666667e+00
INCOME_PER_PERSON,307509.0,9.310588e+04,1.013734e+05,2.812500e+03,47250.000000,75000.000000,1.125000e+05,3.900000e+07
EXT_SOURCE_MEAN,307339.0,5.092512e-01,1.498020e-01,5.939651e-06,0.413648,0.524502,6.228189e-01,8.789034e-01


In [10]:
# share of clients with history per table
train_fe[[c for c in train_fe.columns if c.startswith("HAS_")]].mean().mul(100).round(1)

HAS_BUREAU_HISTORY    85.7
HAS_PREV_HISTORY      94.6
HAS_INST_HISTORY      94.8
HAS_POS_HISTORY       94.1
HAS_CC_HISTORY        28.3
dtype: float64

## 5. Train / validation split (before fitting anything)

In [12]:
X, y = split_xy(train_fe)                       # removes TARGET and SK_ID_CURR
X_train, X_valid, y_train, y_valid = train_valid_split(X, y, test_size=0.2, random_state=42)
X_test, _ = split_xy(test_fe)
print(X_train.shape, X_valid.shape, X_test.shape)
print("default rate  train: %.2f%%  valid: %.2f%%" % (y_train.mean()*100, y_valid.mean()*100))

(246008, 180) (61503, 180) (48744, 180)
default rate  train: 8.07%  valid: 8.07%


## 6. Preprocessing pipeline
- drop columns with more than 60% missing (learned on train)
- numeric: median imputation, then standard scaling
- categorical: fill with "Missing", then one-hot encoding (rare categories grouped, unseen categories ignored)

In [13]:
pipe = build_preprocessor(scale=True)
pipe.fit(X_train)                               # learns from TRAIN only
print("dropped columns:", pipe.named_steps["drop_missing"].cols_to_drop_)

X_train_p = transform_to_frame(pipe, X_train)
X_valid_p = transform_to_frame(pipe, X_valid)   # only transform, never fit
X_test_p = transform_to_frame(pipe, X_test)
print(X_train_p.shape, X_valid_p.shape, X_test_p.shape)
print("missing values left:", X_train_p.isna().sum().sum(), X_valid_p.isna().sum().sum(), X_test_p.isna().sum().sum())

dropped columns: ['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'LIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_AVG', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'FLOORSMIN_MODE', 'LIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MODE', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'FLOORSMIN_MEDI', 'LIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'FONDKAPREMONT_MODE', 'EXT_SOURCE_PROD', 'BUREAU_BB_LATE_SHARE_MEAN', 'CC_BALANCE_MEAN', 'CC_LIMIT_MEAN', 'CC_UTILIZATION_MEAN', 'CC_DRAWINGS_MEAN', 'CC_SK_DPD_MAX']
(246008, 226) (61503, 226) (48744, 226)
missing values left: 0 0 0


## 7. Leakage checks

In [14]:
assert "TARGET" not in X_train.columns and "SK_ID_CURR" not in X_train.columns
assert list(X_train_p.columns) == list(X_valid_p.columns) == list(X_test_p.columns)
assert not set(X_train.index) & set(X_valid.index)          # no row in both sets
print("all leakage checks passed")

all leakage checks passed


In [15]:
# quick look: do the new features relate to TARGET? (train part only)
new_in_model = [c for c in X_train_p.columns if c in new_cols]
pd.concat([X_train_p[new_in_model], y_train], axis=1).corr()["TARGET"].drop("TARGET").sort_values().pipe(
    lambda s: pd.concat([s.head(8), s.tail(8)]))

EXT_SOURCE_MEAN            -0.221785
EXT_SOURCE_MIN             -0.185113
AGE_YEARS                  -0.079467
GOODS_CREDIT_RATIO         -0.064099
YEARS_EMPLOYED             -0.063542
PREV_APPROVED_SHARE        -0.062896
EMPLOYED_TO_AGE_RATIO      -0.057405
DAYS_EMPLOYED_ANOM         -0.046416
PREV_DAYS_DECISION_MEAN     0.043915
BUREAU_DEBT_CREDIT_RATIO    0.049023
BUREAU_ACTIVE_COUNT         0.060203
INST_UNDERPAID_SHARE        0.062477
BUREAU_ACTIVE_SHARE         0.068287
INST_LATE_SHARE             0.070593
PREV_REFUSED_SHARE          0.078208
BUREAU_DAYS_CREDIT_MEAN     0.081026
Name: TARGET, dtype: float64

In [16]:
# optional baseline to confirm the data is model-ready
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_p, y_train)
print("validation ROC-AUC:", roc_auc_score(y_valid, lr.predict_proba(X_valid_p)[:, 1]))

validation ROC-AUC: 0.7676214701395744


## 8. Save processed data and the pipeline

In [17]:
   %pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [ ]:
   import os
   out = project + "outputs/"
   os.makedirs(out, exist_ok=True)
   X_train_p.assign(TARGET=y_train.values).to_pickle(out + "train_processed.pkl")
   X_valid_p.assign(TARGET=y_valid.values).to_pickle(out + "valid_processed.pkl")
   X_test_p.assign(SK_ID_CURR=test_fe["SK_ID_CURR"].values).to_pickle(out + "test_processed.pkl")
   save_pipeline(pipe, out + "preprocessing_pipeline.joblib")
   pd.Series(get_feature_names(pipe)).to_csv(out + "feature_names.csv", index=False, header=["feature"])
   print("saved")

## Summary for Member 3 (modelling)
-   - Load `train_processed.pkl` / `valid_processed.pkl`; the column `TARGET` is the label.
- Final predictions: `test_processed.parquet` (column `SK_ID_CURR` is the id, not a feature).
- For new raw data: `load_pipeline(...)` then `transform_to_frame(pipe, build_features(...)-split_xy)`.
- Tree models: you can rebuild with `build_preprocessor(scale=False)`.

In [19]:
from src.preprocessing import transform_new_applicants

one = {"AMT_INCOME_TOTAL": 150000, "AMT_CREDIT": 400000, "AMT_ANNUITY": 20000,
       "DAYS_BIRTH": -12000, "DAYS_EMPLOYED": -2000,
       "CODE_GENDER": "F", "NAME_EDUCATION_TYPE": "Higher education"}
new_features = transform_new_applicants(pipe, one)
print(new_features.shape, "| missing:", new_features.isna().sum().sum())
print(list(new_features.columns) == get_feature_names(pipe))

(1, 226) | missing: 0
True


In [20]:
   print("train memory (MB):", round(train.memory_usage(deep=True).sum() / 1e6, 1))
   print("duplicate rows:", train.duplicated().sum(), "| duplicate IDs:", train["SK_ID_CURR"].duplicated().sum())
   print("IDs in both train and test:", len(set(train["SK_ID_CURR"]) & set(test["SK_ID_CURR"])))
   print("dtypes:", train.dtypes.value_counts().to_dict())

train memory (MB): 341.0
duplicate rows: 0 | duplicate IDs: 0
IDs in both train and test: 0
dtypes: {dtype('float64'): 65, dtype('int64'): 41, <StringDtype(na_value=nan)>: 16}


In [21]:
from src.preprocessing import build_preprocessor, save_pipeline, load_pipeline, get_feature_names, transform_to_frame
import numpy as np

pipe = build_preprocessor(scale=True).fit(X_train)       # same data, same settings as before
save_pipeline(pipe, out_dir + "preprocessing_pipeline.joblib")

# check 1: the saved file loads
pipe2 = load_pipeline(out_dir + "preprocessing_pipeline.joblib")
# check 2: it reproduces exactly the processed data you already saved
same = np.allclose(transform_to_frame(pipe2, X_valid).values, X_valid_p.values)
print("features:", len(get_feature_names(pipe2)), "| matches saved validation data:", same)

pd.Series(get_feature_names(pipe2)).to_csv(out_dir + "feature_names.csv", index=False, header=["feature"])

features: 226 | matches saved validation data: True


In [22]:
   import os, sys, pandas as pd
   project = os.getcwd() + os.sep
   sys.path.append(project)
   from src.preprocessing import load_pipeline, transform_new_applicants, get_feature_names

   out_dir = project + "outputs/"
   train_p = pd.read_pickle(out_dir + "train_processed.pkl")
   valid_p = pd.read_pickle(out_dir + "valid_processed.pkl")
   X_train_p, y_train = train_p.drop(columns="TARGET"), train_p["TARGET"]
   X_valid_p, y_valid = valid_p.drop(columns="TARGET"), valid_p["TARGET"]
   pipe = load_pipeline(out_dir + "preprocessing_pipeline.joblib")
   print(X_train_p.shape, X_valid_p.shape)

(246008, 226) (61503, 226)


In [23]:
   one = {"AMT_INCOME_TOTAL": 150000, "AMT_CREDIT": 400000, "AMT_ANNUITY": 20000,
          "DAYS_BIRTH": -12000, "DAYS_EMPLOYED": -2000,
          "CODE_GENDER": "F", "NAME_EDUCATION_TYPE": "Higher education"}
   out = transform_new_applicants(pipe, one)
   print(out.shape, "| missing:", out.isna().sum().sum())
   print(list(out.columns) == get_feature_names(pipe))

(1, 226) | missing: 0
True
